In [ ]:
import os

In [ ]:
repo_name = "Image_Captioning_using_DeepLearning"
repo_path = f"/content/{repo_name}"
repo_url = f"https://github.com/divyanshrajput118/{repo_name}.git"

if os.path.exists("/content") and not os.path.exists(repo_path):
    !git clone {repo_url} {repo_path}

In [2]:
import pickle
import json

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tf_keras.preprocessing.sequence import pad_sequences
from tf_keras.models import load_model
from tf_keras.utils import to_categorical
from tf_keras.applications.resnet50 import preprocess_input
from tf_keras.preprocessing.image import load_img, img_to_array
from tf_keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [4]:
def data_generator(img_dict : dict, vocab_size : int, max_length : int):

        while True:
            X1, X2, Y = list(), list(), list()

            img_cnt=0

            for img_id,data in img_dict.items():
                for tokenized_caption in data['tokenized_captions']:
                    for j in range(1, len(tokenized_caption)):
                        cur_seq = tokenized_caption[:j]
                        next_word = tokenized_caption[j]
                        next_word = to_categorical(next_word, num_classes = vocab_size)
                        X1.append(data['extracted_features'])
                        X2.append(cur_seq)
                        Y.append(next_word)

                img_cnt+=1

                if img_cnt == 16:

                    X2_padded = pad_sequences(X2, maxlen=max_length, padding='post')
                    yield (np.array(X1) ,X2_padded), np.array(Y)

                    img_cnt = 0
                    X1, X2, Y = list(), list(), list()
            if X1:
                X2_padded = pad_sequences(X2, maxlen=max_length, padding='post')
                yield (np.array(X1), X2_padded), np.array(Y)
                X1, X2, Y = list(), list(), list()

# @staticmethod
# def save_model(path: Path, model: tf.keras.Model):
#     model.save(path)


def train(train_dict: dict,val_dict: dict, vocab_size:int, max_length:int, main_model):

    steps_per_epoch = len(train_dict) // 16
    if len(train_dict) % 16 != 0: steps_per_epoch += 1
        
    validation_steps = len(val_dict) // 16
    if len(val_dict) % 16 != 0: validation_steps += 1

    train_generator = data_generator(train_dict, vocab_size, max_length)
    val_generator = data_generator(val_dict, vocab_size, max_length)

    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True
    )

    lr_schedule = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,               
        patience=2,               
        verbose=1,
        min_lr=1e-6               
    )

    main_model.fit(
        train_generator,
        steps_per_epoch=steps_per_epoch,
        validation_data=val_generator,
        validation_steps=validation_steps,
        epochs=30,                
        callbacks=[early_stopping, lr_schedule]  
    )


In [ ]:
!pip install tf_keras

In [11]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tf_keras
model = tf_keras.models.load_model("artifacts/prepare_base_model/base_model_updated.h5", compile=False)

In [13]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import shutil, os

# create destination folders
os.makedirs("artifacts", exist_ok=True)
os.makedirs("artifacts/prepare_base_model", exist_ok=True)

src = "/content/drive/MyDrive/sample_artifacts"

shutil.copy(f"{src}/train_img_dict.pkl", "artifacts/train_img_dict.pkl")
shutil.copy(f"{src}/val_img_dict.pkl", "artifacts/val_img_dict.pkl")
shutil.copy(f"{src}/model_tokenizer.pkl", "artifacts/model_tokenizer.pkl")
shutil.copy(f"{src}/base_model.h5", "artifacts/prepare_base_model/base_model.h5")
shutil.copy(f"{src}/base_model_updated.h5", "artifacts/prepare_base_model/base_model_updated.h5")

'artifacts/prepare_base_model/base_model_updated.h5'

In [ ]:
for f in [
    "artifacts/train_img_dict.pkl",
    "artifacts/val_img_dict.pkl",
    "artifacts/model_tokenizer.pkl",
    "artifacts/prepare_base_model/base_model.h5",
    "artifacts/prepare_base_model/base_model_updated.h5",
]:
    print(f, "->", os.path.exists(f))

In [8]:
import sys
import tensorflow.keras.preprocessing.text as text_module

sys.modules['keras.preprocessing.text'] = text_module

import pickle
with open("artifacts/model_tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

In [9]:
import pickle

with open("artifacts/train_img_dict.pkl", "rb") as f:
    train_img_dict = pickle.load(f)

with open("artifacts/val_img_dict.pkl", "rb") as f:
    val_img_dict = pickle.load(f)

In [14]:
train(train_img_dict,val_img_dict,8588,35,model)

Epoch 1/30
284/284 [==============================] - 40s 118ms/step - loss: 4.8077 - accuracy: 0.2305 - val_loss: 4.0126 - val_accuracy: 0.3036 - lr: 0.0010
Epoch 2/30
284/284 [==============================] - 21s 75ms/step - loss: 3.8448 - accuracy: 0.3108 - val_loss: 3.6706 - val_accuracy: 0.3347 - lr: 0.0010
Epoch 3/30
284/284 [==============================] - 22s 77ms/step - loss: 3.5457 - accuracy: 0.3357 - val_loss: 3.5216 - val_accuracy: 0.3489 - lr: 0.0010
Epoch 4/30
284/284 [==============================] - 21s 73ms/step - loss: 3.3551 - accuracy: 0.3509 - val_loss: 3.4186 - val_accuracy: 0.3622 - lr: 0.0010
Epoch 5/30
284/284 [==============================] - 22s 77ms/step - loss: 3.2101 - accuracy: 0.3612 - val_loss: 3.3613 - val_accuracy: 0.3713 - lr: 0.0010
Epoch 6/30
284/284 [==============================] - 21s 74ms/step - loss: 3.0986 - accuracy: 0.3693 - val_loss: 3.3315 - val_accuracy: 0.3754 - lr: 0.0010
Epoch 7/30
284/284 [==============================] - 22s

In [16]:
model

In [17]:
from pathlib import Path
import tensorflow as tf

def save_model(path: Path, model: tf.keras.Model):
    """Saves a TensorFlow Keras model to the specified Path.
    
    Automatically creates any missing parent directories.
    """
    # Convert string to Path object if necessary
    path = Path(path)
    
    # Create parent directories if they don't exist yet
    path.parent.mkdir(parents=True, exist_ok=True)
    
    # Save the model (.h5 or SavedModel format depending on extension)
    model.save(path)
    print(f"✅ Model successfully saved at: {path}")

In [19]:
from pathlib import Path

# Define your paths using Path objects
save_path = Path("/content/drive/MyDrive/sample_artifacts/trained_model.h5")

# Call your function
save_model(path=save_path, model=model)

✅ Model successfully saved at: /content/drive/MyDrive/sample_artifacts/trained_model.h5
